# OGC API Coverages — Domain Set, Range Type, Coverage Slice

Demonstrates OGC API — Coverages against DynaStore.

## What this notebook covers

1. Catalog and collection setup via the `public_open_data` preset.
2. Coverage service landing page and conformance.
3. Domain set introspection (`/coverage/domainset`).
4. Range type / band discovery (`/coverage/rangetype`).
5. Coverage slice with bbox + datetime (`/coverage`).
6. Teardown.

**Env vars:** `DYNASTORE_BASE_URL` (default `http://localhost:8080`),
`DYNASTORE_SYSADMIN_TOKEN` or `DYNASTORE_TOKEN`,
`CATALOG_ID` (default `nb04-coverages-demo`),
`COLLECTION_ID` (default `sentinel2-l2a`).

**Routes verified against** `coverages_service.py` `_register_routes()`:
- `GET /coverages/` → landing page
- `GET /coverages/conformance` → conformance
- `GET /coverages/catalogs` → catalog list
- `GET /coverages/catalogs/{cat}/collections` → 200
- `GET /coverages/catalogs/{cat}/collections/{col}/coverage` → 200
- `GET /coverages/catalogs/{cat}/collections/{col}/coverage/metadata` → 200
- `GET /coverages/catalogs/{cat}/collections/{col}/coverage/domainset` → 200
- `GET /coverages/catalogs/{cat}/collections/{col}/coverage/rangetype` → 200 or 501

In [ ]:
import os
import json
import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")
TOKEN = (
    os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_TOKEN")
    or ""
)

if not TOKEN:
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                TOKEN = _r.json().get("access_token", "")
        except Exception:
            pass

HEADERS = {"Authorization": f"Bearer {TOKEN}"} if TOKEN else {}
client = httpx.Client(base_url=BASE, headers=HEADERS, timeout=120.0)

CATALOG_ID = os.environ.get("CATALOG_ID", "nb04-coverages-demo")
COLL_ID = os.environ.get("COLLECTION_ID", "sentinel2-l2a")
COV_PREFIX = f"/coverages/catalogs/{CATALOG_ID}/collections/{COLL_ID}"

def check(r, label="", expected=(200, 201)):
    msg = f"{label}: {r.status_code}"
    if r.status_code not in expected:
        msg += f" — {r.text[:300]}"
    print(msg)
    assert r.status_code in expected, msg
    return r

# Check whether Coverages extension is mounted
_probe = client.get(f"{COV_PREFIX}/coverage/domainset")
SKIP = _probe.status_code in (404, 501)
if SKIP:
    print(f"NOTE: Coverages endpoint not available (HTTP {_probe.status_code}). Cells will be skipped.")
else:
    print(f"Base URL    : {BASE}")
    print(f"Catalog     : {CATALOG_ID}")
    print(f"Collection  : {COLL_ID}")
    print(f"Cov prefix  : {COV_PREFIX}")

## 1. Catalog setup via `public_open_data` preset

In [ ]:
# Only create the demo catalog when using the default CATALOG_ID.
# If CATALOG_ID points to an existing catalog, skip creation.
_is_demo = CATALOG_ID == "nb04-coverages-demo"
if _is_demo:
    r = client.post("/stac/catalogs", json={
        "id": CATALOG_ID,
        "title": "OGC Coverages Demo",
        "description": "Ephemeral catalog for the OGC API Coverages notebook.",
    })
    check(r, "Create catalog", expected=(200, 201, 409))

    r = client.post(f"/configs/catalogs/{CATALOG_ID}/presets/public_open_data")
    check(r, "Apply preset public_open_data", expected=(200, 201, 409))
    print("Demo catalog ready.")
else:
    print(f"Using existing catalog: {CATALOG_ID}")

## 2. Coverages service landing page and conformance

In [ ]:
if SKIP:
    print("SKIP: Coverages extension not available.")
else:
    r = client.get("/coverages/")
    check(r, "Landing page")
    lp = r.json()
    print(f"Title: {lp.get('title')}")
    print(f"Links: {len(lp.get('links', []))}")

In [ ]:
if SKIP:
    print("SKIP: Coverages extension not available.")
else:
    r = client.get("/coverages/conformance")
    check(r, "Conformance")
    uris = r.json().get("conformsTo", [])
    print(f"Conformance URIs ({len(uris)}):")
    for u in uris[:5]:
        print(f"  {u}")

In [ ]:
if SKIP:
    print("SKIP: Coverages extension not available.")
else:
    r = client.get(f"/coverages/catalogs/{CATALOG_ID}/collections")
    check(r, "List collections")
    data = r.json()
    ids = [c.get("id") for c in data.get("collections", [])]
    print(f"Collections: {ids[:5]}")

## 3. Domain set introspection

`GET /coverages/catalogs/{cat}/collections/{col}/coverage/domainset`
returns the CRS, spatial extent, and grid axes of the coverage.

In [ ]:
if SKIP:
    print("SKIP: Coverages extension not available.")
else:
    r = client.get(f"{COV_PREFIX}/coverage/domainset")
    check(r, "Domain set")
    ds = r.json()
    domain_type = ds.get("type", ds.get("domainType", "unknown"))
    print(f"Domain type : {domain_type}")
    envelope = ds.get("generalGrid", {}).get("envelope", ds.get("envelope", {}))
    if envelope:
        print(f"Extent      : {envelope.get('low', '?')} → {envelope.get('high', '?')}")
        print(f"CRS         : {envelope.get('srsName', envelope.get('crs', '?'))}")
    else:
        print("Envelope not present in response.")

## 4. Range type / band discovery

`GET /coverages/catalogs/{cat}/collections/{col}/coverage/rangetype`
returns band names, unit-of-measure, and data types.
Drivers without `INTROSPECTION` capability return **501** — handled gracefully.

In [ ]:
if SKIP:
    print("SKIP: Coverages extension not available.")
else:
    r = client.get(f"{COV_PREFIX}/coverage/rangetype")
    assert r.status_code in (200, 501), (
        f"rangetype: unexpected status {r.status_code}: {r.text[:300]}"
    )
    if r.status_code == 200:
        rt = r.json()
        fields = rt.get("field", [])
        print(f"Bands ({len(fields)}):")
        for fld in fields:
            uom = fld.get("uom", {}).get("code", "-")
            dtype = fld.get("dataType", "-")
            print(f"  {fld.get('name', '?'):20s}  uom={uom}  dtype={dtype}")
    else:
        print("rangetype: HTTP 501 — driver does not implement INTROSPECTION.")

## 5. Coverage metadata

`GET /coverages/catalogs/{cat}/collections/{col}/coverage/metadata`

In [ ]:
if SKIP:
    print("SKIP: Coverages extension not available.")
else:
    r = client.get(f"{COV_PREFIX}/coverage/metadata")
    check(r, "Coverage metadata", expected=(200, 204))
    if r.status_code == 200:
        meta = r.json()
        print(f"Metadata keys: {list(meta.keys())[:8]}")

## 6. Coverage slice — bbox + datetime

`GET /coverages/catalogs/{cat}/collections/{col}/coverage`
with `bbox` and `datetime` parameters.

In [ ]:
if SKIP:
    print("SKIP: Coverages extension not available.")
else:
    BBOX = "37.0,8.0,43.0,15.0"  # Ethiopia highlands: minLon,minLat,maxLon,maxLat
    DATETIME = "2024-02-01T00:00:00Z/2024-02-28T23:59:59Z"

    r = client.get(
        f"{COV_PREFIX}/coverage",
        params={"bbox": BBOX, "datetime": DATETIME},
    )
    check(r, "Coverage slice")
    ct = r.headers.get("content-type", "")
    print(f"Content-Type : {ct}")
    print(f"Response size: {len(r.content)} bytes")

    if "json" in ct.lower():
        body = r.json()
        assert "type" in body, f"CoverageJSON must have 'type' key; got keys: {list(body.keys())}"
        assert "domainset" in body or "domain" in body, (
            f"CoverageJSON must include 'domainset' or 'domain'; got: {list(body.keys())}"
        )
        print(f"Coverage type: {body.get('type')}")

In [ ]:
if SKIP:
    print("SKIP: Coverages extension not available.")
else:
    # Out-of-extent bbox must not produce 4xx — spec requires an empty valid response
    r = client.get(
        f"{COV_PREFIX}/coverage",
        params={"bbox": "-30.0,60.0,-20.0,70.0", "datetime": DATETIME},
    )
    assert r.status_code < 400, (
        f"Out-of-extent bbox must not produce 4xx. Got {r.status_code}: {r.text[:200]}"
    )
    print(f"Out-of-extent bbox → HTTP {r.status_code} (expected 200 with empty/null coverage)")

## Teardown

In [ ]:
if _is_demo:
    r = client.delete(f"/stac/catalogs/{CATALOG_ID}")
    print(f"Delete catalog: {r.status_code}")
else:
    print("Existing catalog left intact.")

client.close()
print("Teardown complete.")